Microbialite Simulator 2D 

This code is setup up in a single notebook which can perform all the functions required in the publication. The intial cells define the model and it's rules while the later cells can be used to sample the parameter space, generate single objects and morphospaces 

In this code, there are @everyone begin... end statements surrounding blocks so that functions are distrubuted such that when doing paramter sweeps this can run on each core.

In [ ]:


#Importing all required packages



#import Pkg

# List of required packages
#These Packages are not version pinned as the main syntax should be robust
#packages = ["LinearAlgebra", "Random", "StatsBase", "GLMakie"]

# Install any packages that aren't already present
#for p in packages
#    Pkg.add(p)
#end

#Version Pin Agents.jl
#Pkg.add(Pkg.PackageSpec(name="Agents", version="7.0.3"))






#Loading so that we can run parameter sweeps in parrallel
using Distributed
@everywhere begin

# Import the Agents package for agent-based modeling functionality
# Import LinearAlgebra for matrix and vector operations
using Agents, LinearAlgebra

#Need for analysis of perodicity and structure plus saving data
using FFTW
using Images, ImageMorphology
using DataFrames, Statistics, CSV 


# Import Random for generating random numbers and sequences
using Random

# Import StatsBase for statistical functions and tools
using StatsBase
    

#Needed for plotting and making videos of dyanmics
#When mixing plotting librarires might need to name the library when calling to prevent errors
using CairoMakie, Plots, Printf
using FileIO, ImageMagick
using PyPlot
using LaTeXStrings #If you want to use LaTex font on figures

    
    
#Used for data saving and some measuremnts    
using HDF5

using FileIO, Images, ImageTransformations

using CodecZlib

#Utilities for Uniqute File Names 
using SHA, Dates

#Utlities
using ProgressMeter, BenchmarkTools


#Needed for Parameter Sweeps 
using LatinHypercubeSampling
using StatsBase 




#helper function for counting
function count_at_least(iter, threshold)
    count_val = 0
    for _ in iter # Iterate through the collection
        count_val += 1
        if count_val > threshold
            return true # Found more than the threshold, no need to count further
        end
    end
    return false # Did not find more than the threshold
end

#helper function for finding max height
function find_max_height(A)
    num_rows = size(A, 2)
    for r in num_rows:-1:1  # Iterate from the last row upwards
        if any(x != 0 for x in A[:, r]) # Check if any element in the row is non-zero
            return r
        end
    end
    return 0 # If no non-zero values are found in the entire array
end




    
    

end

The cell below intialises the agents of two classes, class one which are named Sediment representing the sediment particles which are introduced to the simulation. While class two called Precipitate represts the aggreation affects of the mirocobial mat which lead to precipiation. The naming convention should not suggestion that this is actually a simulation of a single micorbe or realistic mircobial mat dynamics. The main focus is on precipitation as this is what would be preserved at larger scales. 


In [ ]:
@everywhere begin

#Define the Agents types in the simulations
#Set the dimension and Space type with GridAgent{2}, {2} for 2d and {3} for 3d 

# Define a Sediment agent type
# This agent represents sediment particles in the simulation
@agent struct Sediment(GridAgent{2})
    # No additional properties defined for sediment
end

# Define a Precipitate agent type
# This agent represents an approximation of microbial mats that can grow on surfaces  or precipitate processes
@agent struct Precipitate(GridAgent{2})
   
end

end

Update Rules for Sedimenting agents

In [ ]:
@everywhere begin


#Agent Stepping function using Julia multiple Dispatch, same function name for each agent type 
#Function which acts on the Sediment particles making them move or stick

#Preference for left/right motion in order [left,right] 
#Modify for 3D
const weights_side= pweights([0.5,0.5])

#Preference for down/up motion in order [down,up] 
#Want probability down to be greater than up so particles on average drift down 
const weights_vert = pweights([0.9,0.1])




function agent_step!(sediment::Sediment, model)




    #Call RNG from model 
    rng = abmrng(model)
    
    #Get the position of the agent
    agent_pos = sediment.pos

    #Extract the attraction and stability distance from the model

    attraction_dist = model.attraction_dist
    stability_dist = model.stability_dist
        
    #Find the nearby locations within the attaction distance
    nearby = nearby_positions(agent_pos, model, attraction_dist)

    
    #Find the locations which are solid to potentially stick near
    solid_nearby_locs = [loc for loc in nearby if model.solid[loc...] != 0]

    #If any solid locations were found, shuffle them, to prevent bias in model
    if !isempty(solid_nearby_locs)
    # Use shuffle! for in-place shuffling
        shuffle!(rng, solid_nearby_locs)
    end


    
    #Start Looping through solid locations for places to stick 
    for loc in solid_nearby_locs


      
            #Search within stabilty distance of identified location
            stability_search = nearby_positions(loc, model, stability_dist)

            #Create a containers for locations and weights
            targets_loc = Tuple{Int, Int}[]
            weights = Float64[]

            # Counter for number of valid targets
            ntargets = 0

           #Loop through nearby positions
            for place in (stability_search)

            
                #Check if potential location is empty 
                if model.solid[place...]==0

                    #Add location to list of potential places 
                    push!(targets_loc, place)


                    
                    #Loop through neighbours to check if they are full, want to stick near solid objects 
                    local_positions = nearby_positions(place, model, 1)
                    local_weight = 0

                    for p in local_positions
                        #Update local weights with count of non-zero neighbours
                        #Can update this function have any weight rule
                        local_weight = local_weight + (model.solid[p...] >0)
                    end
                    #Add local_weights to weights list
                    push!(weights, local_weight)
                    #weights[ntargets] = local_weight
                    ntargets += 1
                end 
            end 


            #Check weights list is not empty 
            if ntargets > 0

                #Convert weights to probablities
                probabilities = weights ./ sum(weights)
                
                #Select a location with given probability using Julia Sample function
                choice = sample(rng, targets_loc, pweights(probabilities))



                
                #Set the location to non-zero value represting a solid, actual value represents colour or another property for visualisation
                model.solid[choice...] = 2


            
                #Remove the sediment agent
                remove_agent!(sediment, model)
               
                #Check if space is empty and add Precipitate agent according to recolonisation probability 
                if isempty(choice, model) && (model.recolonisation_prob > rand(rng))
                    add_agent!(choice, Precipitate, model)
                    end
                #End the function evaluation here if managed to find a place to stick 
                return
            end 
        end

    #If a place to stick is not found move onto the movement steps 

 

   

    #Function to pick walk direction and set biases of direction
    #Weights should sum to 1 as they are probabilities
    #The size of the steps could also be drawn from a distubtion but constant step size for simplicty


    
    #Pick the direction by sampling from [-1,1] using the given weights
    dir_side = sample(rng, [-1,1], weights_side)

    dir_vert = sample(rng, [-1,1], weights_vert)

    #Extract the stepsize from model in each direction
    #This could be drawn from a distubtion but should an integer for the grid spacing 
    
    down_speed = model.downspeed
    sideways_speed = model.sidespeed

    
    #Execute the particle step with the Agents.jl walk function
    #Checking the space to move to is empty
    walk!(sediment, (dir_side*sideways_speed,dir_vert*down_speed), model;
        ifempty = true
        )

    #Finally do some checks to see if agent needs to be removed


    #First check the agent is not at the top of the grid for array bounds before the check 
    if agent_pos[2] != model.height

    #Check if the agent has either reached the bottom of the grid or walked into the solid and remove if this is the case     
    if agent_pos[2]== 1 || model.solid[agent_pos[1], agent_pos[2] + 1] != 0
        #remove the agent
        remove_agent!(sediment, model)
        #Terminate the function
        return
    end
    end  


    

    

end


end

Update function for precipiating agents.  

In [ ]:
@everywhere begin
#Multiple Dispatch Function to step the Precipitate Agents




function agent_step!(precipitator::Precipitate, model)

    #Call RNG from model 
    rng = abmrng(model)
    

    #Start by checking if the agent should be randomly removed
    #Probablity that the agent is randomly removed
    if model.remove_probs > rand(rng)
    #Delete agent and terminate the loop
    remove_agent!(precipitator, model)
    return
    end
    
    #Get the position of the agent
    agent_pos = precipitator.pos


    #Get the direct neeighbours of the agent
    neighbours = nearby_positions(agent_pos, model, 1)
    #Initialise counts of the neighbours and if they are full 
    solid_count::Int = 0
    total_count::Int = 0 
    
    blocking_count::Int = 0
    
    #Create containers for locastions agent can grow to and weights
    empty = Tuple{Int, Int}[]
    weight = Float64[]

    #Loop throgh the neighbours 
    for place in (neighbours)

        #Increase the total_count of all neighbouring spaces by 1
        total_count = total_count + 1

        #Check if the neighbour is solid 
        if model.solid[place...] >0
        #Count number of solid neighbours
        solid_count = solid_count + 1



        #Update the count of blocking neighbours         
        if place[2] >= agent_pos[2] 
        blocking_count += 1
            end

        #If space is not filled compute probablity to grow there     
        else 

            #Add location to empty space list
            push!(empty, place)

            
            #Compute the height difference assuming Manhattan or Euclidean distance here
            height_difference = place[2] - agent_pos[2]

            # Map height difference -1, 0, 1 to array indices 1, 2, 3
            #Array is set in model intialisation and gives preference for growing downwards, sideways or upwards
            weight_index = height_difference + 2

            

            #Add weights to weight list 
            push!(weight, model.precip_weights_down_side_up[weight_index])
        end 
    
        
    end
    

    


    #If all the neighbours of the precipitator are solid remove it. 
    if solid_count == total_count
    remove_agent!(precipitator, model)
        return


    

        
    #Allow precipitator growth
    else 
        #Ensure Growth is actaully favourable and not just default emptry region
        weight_sum = sum(weight)
       if isempty(empty)==false && (weight_sum > 1e-9)

            #Convert weights to probability
            weight = weight/weight_sum 

            #Sample weights 
            choice = sample(rng, empty, pweights(weight))


    #Compute exposure factor if conical behaviour is being induced            
        
     exposure_factor = 1.0 / (blocking_count + 1)^model.conical_scale 


    #Set flag to true for overhang prevention logic  
                
    is_stable_location = true 


    #Begin Overhang prevention logic
                

    if model.prevent_overhang == true 


    target_x, target_y = choice

    is_on_floor = (target_y <=2)

    # Check the spot directly below the target
    has_vertical_support = (target_y > 1) && (model.solid[target_x, target_y - 1] > 0)

    # Check the spots diagonally below (left-down and right-down)
    # This allows the cone to widen by 1 pixel for every 1 pixel of height (45 degrees)
    has_diagonal_support = (target_y > 1) && (
        (target_x > 1 && model.solid[target_x - 1, target_y - 1] > 0) || 
        (target_x < model.width && model.solid[target_x + 1, target_y - 1] > 0)
        )

    
    #reset flag to required state
    is_stable_location = is_on_floor || has_vertical_support || has_diagonal_support


                end 



     
        
    #If the space is empty check the 
    if is_stable_location && isempty(choice, model)  && (model.precip_probs*exposure_factor > rand(rng))

        #Fill solid with Int to represent properties for visulisation
        model.solid[choice...] = 3
       
        add_agent!(choice, Precipitate, model)
    end
        end
    end
end



end

Code to step forward the world and add more sediment. Has some options which can be modified at runtime and other hard-coded behaviours which could be changed.

In [ ]:
@everywhere begin


#Function which steps the world forwards and adds new sediment to the simulation 
function world_step!(model)
    #Call RNG from model 
    rng = abmrng(model)
    
    #Can in theory make any of the parameters a function of time at this point
    #Any parameter could be called and updated here if you like


    #sed_ment_rate = model.sediment_amplitude

    t= abmtime(model) 

    # Precompute angular frequencies
    sediment_ω = 2π / model.period_control_T
    precipitator_ω  = 2π / (model.precip_period_scaling * model.period_control_T)

    # Helper inline functions
    periodic_value(mode::Symbol, x) = mode === :max_cos  ? max(0, cos(x)) :
                                  mode === :abs  ? abs(cos(x))   :
                                  mode === :cos2 ? cos(x)^2      :
                                  mode === :constant ? 1.0           :
                                  error("Unknown periodic mode: $mode")


    sediment_amplitude = round(Int, model.sediment_amplitude *
                      periodic_value(model.sediment_periodic_mode, t * sediment_ω))

    model.precip_probs = model.precip_amplitude * periodic_value(model.precip_periodic_mode,(t + model.precip_period_offset) * precipitator_ω)



    #Set how near the edges of the grid particles should be released
    min_col::Int = 1
    max_col::Int = model.width 

    

    if mod(t, 10)==0
            model.max_height = find_max_height(model.solid)
            #println(model.max_height)
        end



    # Compute a safe random offset
    down_offset = model.downspeed > 1 ? rand(1:(model.downspeed - 1)) : 0

    input_row = min(model.height - 5, 10*model.downspeed + model.max_height) - down_offset
    

    #Get range of potential releases 
    candidate_cols = min_col:max_col

    input_row_offsets = input_row .+ (-5:5)
    potential_positions = [(col, row) for col in candidate_cols for row in input_row_offsets]

    #Check for valid release positions
    #potential_positions = [(col, input_row) for col in candidate_cols]

    valid_positions = [
        pos for pos in potential_positions
        if model.solid[pos...] == 0 && isempty(pos, model)
    ]


    #Get number of particles to add if space is shorter than expected number 
    num_to_add = min(sediment_amplitude, length(valid_positions))
    
    if num_to_add > 0
        # Sample 'num_to_add' unique positions from 'valid_positions'
        # Using shuffle! and taking the first num_to_add is efficient if num_to_add << length(valid_positions)
            #shuffle!(rng, valid_positions)[1:num_to_add]
        chosen_positions = shuffle!(rng, valid_positions)[1:num_to_add]

        # Add sediment agents at the chosen locations, should be empty from previous checks
        for pos in chosen_positions
            add_agent!(pos, Sediment, model)
        end
    end
end 



end

Code to control parameters and setup model. Some intial layouts are hard coded so this will need to be changed if the model size is varied too much. 

In [ ]:
@everywhere begin




#Using a Mutable Struct so that code runs faster due to type information provided
#All parameters can be modifed as the simulation runs however they need to respect the type defintions
#All parameter defaults are listed below, with explaination given 

Base.@kwdef mutable struct WorldParameters
    #Width of the simulation
    width::Int = 1000
    #Height of the simulation
    height::Int = 400
    #Type of intial conditions can be :base_line, :rough, :localised, :mound, :triangle details can be changed by function modfiaction
    initial_solid_setup_type::Symbol = :base_line
    #Speed that particles step in the vertical direction
    downspeed::Int = 3
    #Speed that particles step in the horizontal direction, want to keep as 1 so particles can hit ever point if doing single point release
    sidespeed::Int = 1
    #Probability that a precipitator agent grows
    precip_amplitude::Float64 = 0.08
    #This precip_probs is dyanmically update through the simulations if using periodic precipitator activty
    precip_probs::Float64 = 0.8
    #Probability that a precipitator agent is removed at each time step
    remove_probs::Float64 = 0.02
    #Preferences for precipitator growth directions
    precip_weights_down_side_up::Vector{Int} = [0, 1, 1]
    #Sediment Rate maximum value
    sediment_amplitude::Int = 50
    #Parameter which is related to the period of the sediment rate
    period_control_T::Float64 = 200
    #Parameter which sets how to scale the precipitator growth period relative to the sedimentation rate
    precip_period_scaling::Float64 = 1
    #Parameter which sets how to offset the precipitator growth osciliations relative to the sedimentation rate
    precip_period_offset::Float64 = 50 
    #Stability distance of the DLA
    stability_dist::Int = 15
    #Attraction distance of the DLA
    attraction_dist::Int = 15
    #How much to weight growing out of the tips to create conical strcuture,
    #Set to zero for no-bias and higher, values for more bias, this will rescale growth so growth rates need to increase in tandem
    conical_scale::Float64 = 0
    #Intialisation of the solid matrix to store the state
    solid::Matrix{UInt8} = zeros(0, 0)
    #Choice of the distance matrix
    distance_metric::Symbol = :euclidean#options should be :euclidean, :manhattan for physical behaviour; :chebyshev will also work but beware of corners
    initial_precipitator_density::Float64 = 0.9 #Propotion of intial substrate which contains a precipitator
    #Recolonisation Probability - Pararmeter which sets how likely micorobe to appear at site of newly landed sediment
    recolonisation_prob::Float64 = 0.5
    max_height::Int = 100
    #Flags to choose behaviour of perodic function
    sediment_periodic_mode::Symbol = :abs # e.g. :max_cos, :cos2, :abs, :constant
    precip_periodic_mode::Symbol =  :abs # e.g. :max_cos, :cos2, :abs, :constant
    #Flag to set if overhanging horizontal behaviour is allowed, prevents mushroom type strctures  
    prevent_overhang::Bool = false
    
end



#
function makeworld(
initial_params::WorldParameters;
seed::Union{Nothing, Int} = nothing, rng::AbstractRNG = Random.default_rng()
)


space = GridSpaceSingle((initial_params.width, initial_params.height); 
                        periodic = (true,false), 
                        metric = initial_params.distance_metric)

# Initialise solid matrix
solid_setup = zeros(UInt8, initial_params.width, initial_params.height)

# Determine bottom layer / roughness
if initial_params.initial_solid_setup_type == :base_line
    solid_setup[:, 1] .= 1

elseif initial_params.initial_solid_setup_type == :rough
    h = round.(Int, 5*sin.(collect(1:initial_params.width)*(2*pi)/300) + abs.(1*randn(initial_params.width)))
    for a in 1:initial_params.width
        solid_setup[a, 1:h[a]] .= 1
    end

elseif initial_params.initial_solid_setup_type == :localised
    #Integer Division
    side = initial_params.width ÷ 2
    #Specfic region for overhangs       
    solid_setup[(side -5):(side + 5), 1] .= 1
       

elseif initial_params.initial_solid_setup_type == :mound
    # Define the center and width of your mound
    #Integer Division        
    center_x = initial_params.width ÷ 2
    mound_half_width = 75  # Total width of 150
    mound_height = 10      # Height of the center peak in pixels

    for x in (center_x - mound_half_width):(center_x + mound_half_width)
        # Ensure we are within grid bounds
        if 1 <= x <= initial_params.width
            # Parabolic equation: y = H * (1 - (dist/W)^2)
            # This creates a smooth mound instead of a sharp triangle
            dist = abs(x - center_x)
            h = round(Int, mound_height * (1 - (dist / mound_half_width)^2))
            
            # Fill from bottom (1) up to calculated height (h)
            # Use max(1, h) to ensure there is at least a base line
            for y in 1:max(1, h)
                solid_setup[x, y] = 1
            end
        end
    end


elseif initial_params.initial_solid_setup_type == :triangle

    #Integer Division
    center_x = initial_params.width ÷ 2
    tri_half_width = 15  
    tri_height = 100      

    # Only loop through the range of the triangle itself
    # This leaves solid_setup as 0 everywhere else
    for x in (center_x - tri_half_width):(center_x + tri_half_width)
        # Ensure we stay within the horizontal bounds of the simulation
        if 1 <= x <= initial_params.width
            dist = abs(x - center_x)
            
            # Linear decay for a sharp point
            h = Int(ceil(tri_height * (1 - (dist / tri_half_width))))
            
            # Fill from y=1 up to h. If h is 0 (at the very edges), nothing is filled.
            for y in 1:h
                solid_setup[x, y] = 1
            end
        end
    end

            
else
    solid_setup[:, 1] .= 1
end

initial_params.solid = solid_setup

# Create ABM
model = StandardABM(
    Union{Sediment, Precipitate}, 
    space; 
    properties = initial_params,
    rng, 
    agent_step! = agent_step!, 
    model_step! = world_step!,
    warn = false,
    scheduler = Schedulers.Randomly()
)

# Add precipitators based on initial_precipitator_density
for x in 1:model.width
    for y in 1:model.height
        if solid_setup[x, y] != 0 && rand(rng) < initial_params.initial_precipitator_density
            add_agent!((x, y), Precipitate, model)
        end
    end
end

return model

    end
end

# --- Usage  and test set-up 
# 1. Create a parameter instance- This can be defaulted or can modify individual parameters
my_params = WorldParameters(sediment_amplitude=20, precip_probs=0.9, width=1000)

# 2. Initialise the model, rng is not required but can be added
model = makeworld(my_params)




The below cell contains some helper functions whoch are needed in the remainder of the code.

In [ ]:


@everywhere begin



function stopping_function(model, t)

    
    # The condition that returns the final boolean
    condition_met = model.max_height > ( model.height - 50) || t >= num_steps
    
    
    return condition_met
end


#Function to colour the agents when plotting them 
function agent_color(agent)
    if agent isa Precipitate
        return :green # Color for Precipitate agents
    elseif agent isa Sediment
        return :gray # Color for Sediment agents
    else
        return :gray # Default color for any other unexpected types
    end
end
    

end

SINGLE RUN CODE: Code to run and a plot a single run of the model to create a visualisation or to test parameters.

In [ ]:
@everywhere begin


#Function to colour the agents when plotting them 
function agent_color(agent)
    if agent isa Precipitate
        return :green # Color for Precipitate agents
    elseif agent isa Sediment
        return :gray # Color for Sediment agents
    else
        return :gray # Default color for any other unexpected types
    end
end

#Function which extract the model propeties for plotting
get_solid_setup(model) = model.solid

#Plotting arguments
plotkwargs = (
    agent_size = 3, 
    heatarray = get_solid_setup,
    heatkwargs = (colorrange = (0, 3), colormap = :gnuplot),
    agent_color = agent_color,
      
    #colorbar = false, 
    add_colorbar = false,
    
)

end

    
 my_params = WorldParameters( sediment_amplitude= 100, downspeed= 3, width = 2000,
    height = 2*950, recolonisation_prob =0.99 , period_control_T=2000, precip_amplitude=0.08, stability_dist = 40, attraction_dist = 20, 
sediment_periodic_mode = :max_cos,
    precip_periodic_mode=  :max_cos, precip_weights_down_side_up = [1,1,3], conical_scale= 0,
    distance_metric = :euclidean, #options should be :euclidean, :manhattan, :chebyshev
    initial_solid_setup_type = :baseline,  prevent_overhang = false
)

model = makeworld(my_params, seed=1234)




num_steps = 1*10000 # The total number of steps you want to simulate

# Initialize the progress bar
# @showprogress creates and manages the progress bar for the loop
@showprogress "Simulating steps..." for i in 1:num_steps
    # Perform a single step of the simulation

    step!(model, stopping_function) 
    
end


fig, _ = abmplot(model; figure = (size = (600, 600),),  plotkwargs...)







function find_max_height(A)
    num_rows = size(A, 2)
    for r in num_rows:-1:1  # Iterate from the last row upwards
        if any(x != 0 for x in A[:, r]) # Check if any element in the row is non-zero
            return r
        end
    end
    return 0 # If no non-zero values are found in the entire array
end


max_height = find_max_height(model.solid)

println(max_height)



#save("bumpy_comparison_2.png", fig, dpi =300)
fig


MORPHOSPACE CODE: Cells to loop though and plot a morphospace of a given range of parameters. Plot should automatically reshape but only up to a certain size. 

In [ ]:
#Code to loop through parameters 
#Want to garbage collect manually when potentially mixing parameter sweeps 
GC.gc() 

@everywhere begin

#Helped function
get_solid_setup(model) = model.solid




#Check number of threads, may need to restart kernel if this is not what you expect
println("Number of Threads: make sure set to higher than 1 for parrellisation of parameter sweep")
println(Threads.nthreads())


#Set of paramters to sweep through
#Some options givej
all_params_to_sweep = Dict(
    
    
    :stability_dist => [5, 15, 25, 35],

    :attraction_dist => [5, 15,25, 35],


    #:conical_scale => [0,1,2,3],


    #:precip_amplitude => [0.0001, 0.1, 0.2, 0.3],

    #Conical
    #:precip_amplitude => [ 0.1, 0.2, 0.3, 0.4],

        
    #:recolonisation_prob => [0.001,0.2,0.5,0.9],
    
    #For morphology internal just change this value and all else default
    #:period_control_T => [1,200,500,1000],
    
    
   #:initial_solid_setup_type => [ :base_line, :localised, :rough],
    #:precip_amplitude => [0.0001, 0.1, 0.2, 0.3],
    #:remove_probs => [0.02, 0.1,0.2],
    #:precip_weights_down_side_up => [[1,1,1], [5,5,1],[1,1,4], [1,1,100]],
    #:eps_range => [2,5,10, 20],
    #:eps_threshold => [1,5,10,100],
    #:sediment_amplitude => [3,20,25,50],
    #:downspeed => [3,5,9,30]
)


#Create an object which contains all the parameter combinations by taking the cartesian product Iterators.product(param_values_arrays...)    
param_combinations = []
param_keys = collect(keys(all_params_to_sweep))
param_values_arrays = [all_params_to_sweep[k] for k in param_keys]

for combo_vals in Iterators.product(param_values_arrays...)
    combo_dict = Dict(param_keys[i] => combo_vals[i] for i in 1:length(param_keys))
    push!(param_combinations, combo_dict)
end

end

@everywhere begin



println("Number of Combinations scanning")
println(size(param_combinations))

#Array to hold plotting data
plot_data = []

#Object to make the data adding thread safe otherwise chance of memory access errors
plot_data_lock = ReentrantLock()

   
end 


num_steps = 30000 

#Loop through parameters 
#If you want conical need to turn up growth rate as it rescaled by the blocking effect
@showprogress Threads.@threads for p_combo in param_combinations

    #Setup paramters- onces to be scanned to called p_combo
    #Other ones can be left as default or vaired
    my_params = WorldParameters( 
        #sediment_amplitude= 50,
        width = 700,
    height = 1000, period_control_T= 1000,   
        #initial_solid_setup_type = :localised,
        initial_solid_setup_type  = :base_line,
        #initial_solid_setup_type  = :triangle,
        precip_weights_down_side_up = [1,1,3],
        
        recolonisation_prob =0.99, sediment_periodic_mode = :max_cos ,
    precip_periodic_mode=  :max_cos,
         #conical_scale= 0, prevent_overhang = false
        ;p_combo...)



        #concical Morphospace
       # my_params = WorldParameters( 
     #   sediment_amplitude= 25,
    #    width = 700,
   # precip_amplitude = 0.3,
   # height = 1000, period_control_T= 1000,   
        #initial_solid_setup_type = :localised,
        #initial_solid_setup_type  = :base_line,
   #     initial_solid_setup_type  = :triangle,
   #     precip_weights_down_side_up = [1,1,10],
   #     recolonisation_prob =0.99, remove_probs =0, sediment_periodic_mode = :max_cos ,
  # precip_periodic_mode=  :max_cos,
   #       prevent_overhang = true
   #    ;p_combo...)


    #Set-up model with or without seed 

    model = makeworld(my_params, rng = Xoshiro())

    num_steps = 30000 
    
    #Step the model
    step!(model, stopping_function)

    
    #Add to the data array in a thread safe way
    lock(plot_data_lock) do
    push!(plot_data, (p_combo, model.solid))


 

    println(p_combo)
   








        
end


  
end



#Run the garbarge collection again
GC.gc() 



Morphospace plotting code, using PyPlot and the Matplotlib backend for more control over plot arrangements

In [ ]:
GC.gc() 

 # Recommended for easier LaTeX text input

# 1. Enable LaTeX rendering for all text in the plot
PyPlot.matplotlib.rcParams["text.usetex"] = true

# 2. Choose your desired font family
# Common choices for LaTeX-like appearance:
# 'serif' (default Computer Modern Roman)
# 'sans-serif' (default Computer Modern Sans Serif)
# 'monospace' (default Computer Modern Typewriter)
PyPlot.matplotlib.rcParams["font.family"] = "sans-serif"



variables = collect(keys(plot_data[1][1]))

println(variables)

#  Define row and column parameters for the grid
ROW_PARAM = variables[2]
COL_PARAM = variables[1]

#  Get unique, sorted values for row and column parameters
# We rely on all_params_to_sweep to define the possible values


row_values = reverse(sort(all_params_to_sweep[ROW_PARAM]))
col_values = sort(all_params_to_sweep[COL_PARAM])


# Create a dictionary for easy lookup of plot data
# This maps (row_param_value, col_param_value) -> solid_data
plot_data_map = Dict{Tuple{eltype(row_values), eltype(col_values)}, Any}()

for (p_combo, solid) in plot_data
    if haskey(p_combo, ROW_PARAM) && haskey(p_combo, COL_PARAM)
        row_val = p_combo[ROW_PARAM]
        col_val = p_combo[COL_PARAM]
        plot_data_map[(row_val, col_val)] = solid
    else
        @warn "Parameter combination did not contain expected ROW_PARAM or COL_PARAM. Skipping." p_combo
    end
end




# Create the Figure directly
rows = length(row_values) 
cols = length(col_values) 


# --- PyPlotting starts here ---
using PyPlot



# Determine common color range for all heatmaps
# This ensures consistent color scaling across all subplots
all_solid_data = [data for (_, data) in plot_data]
if !isempty(all_solid_data)
    # Flatten all data and find min/max
    min_val = minimum(minimum.(all_solid_data))
    max_val = maximum(maximum.(all_solid_data))
else
    min_val, max_val = 0.0, 1.0 # Default if no data
end

# Adjust figure size for better visualization of 3x3 grid
# You might need to tweak figsize based on your actual data dimensions and desired output
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 3), sharex=true, sharey=true, dpi =100)

# Handle cases where subplots returns a 1D array or a single Axes object
if rows == 1 && cols == 1
    axes_arr = [axes]
elseif rows == 1 || cols == 1
    axes_arr = reshape(axes, rows, cols) # Ensure it's a 2D array
else
    axes_arr = axes
end

# Iterate and plot heatmaps
for i in 1:rows
    row_val = row_values[i]
    for j in 1:cols
        col_val = col_values[j]
        ax = axes_arr[i, j] # Access the correct subplot axis

        key = (row_val, col_val)

        if haskey(plot_data_map, key)
            solid_data = plot_data_map[key]
            # PyPlot's imshow is similar to Matplotlib's imshow for heatmaps
            # 'cmap' can be changed (e.g., "viridis", "plasma", "magma", "cividis")
            # origin="lower" or "upper" depending on how your data is oriented
            # You might need to transpose solid_data if its dimensions are (height, width)
            # and imshow expects (width, height)
            im = ax.imshow(transpose(UInt8.(solid_data)), cmap="gnuplot", vmin=min_val, vmax=max_val, origin="lower", rasterized=true)
            
            #if i != rows
            ax.set_xticks([])
            #end
            ax.set_yticks([]) # Hide y-axis ticks
        else
            # If data is missing for a combination, show a message
            ax.text(0.5, 0.5, "Data Missing", horizontalalignment="center", verticalalignment="center",
                    transform=ax.transAxes, fontsize=12, color="red")
            ax.set_facecolor("lightgray") # Indicate missing data
            ax.set_xticks([])
            ax.set_yticks([])
        end
    end
end

# Add row and column labels
# Set overall row labels on the left
for i in 1:rows
    row_val = row_values[i]
    ax = axes_arr[i, 1] # First column subplot
    ax.set_ylabel("$(row_val)", fontsize=16, rotation=0, ha="right", va="center", weight="bold")
    ax.get_yaxis().set_label_coords(-0.03, 0.5) # Adjust position
end

# Set overall column labels on the top
for j in 1:cols
    col_val = col_values[j]
    ax = axes_arr[rows, j] # First row subplot
    ax.set_xlabel("$(col_val)", fontsize=16, ha="center", va="top", weight="bold") # Use set_xlabel
    ax.xaxis.set_label_coords(0.5, -0.03) # Adjust position relative to the axi
end


label_dict = Dict(
    :precip_amplitude => "Precipitation Amplitude (\$A_{precip}\$)",
    :precip_frac_total => "Precipitate Solid Fraction",
    :attraction_dist => "Attraction Dist.",
    :filled_frac_SNR => "Filled Fraction SNR",
    :filled_frac_freq => "Filled Fraction Freq.",
    :filled_frac_mean => " Filled Fraction Mean",
    :filled_frac_std => "Filled Fraction Std. Dev.",
    :gzip_compression_ratio => "Compression Ratio",
    :max_height => "Max. Height",
    :mean_height => "Mean Height",
    :roughness_cv => "Roughness (Coefficent of Variation)",
    :roughness_local => "Roughness (Local)",
    :sed_frac_total => "Sediment Solid Fraction",
    :stability_dist => "Stability Dist.",
:period_control_T => "Period Control Parameter (\$T\$)",
:sediment_amplitude => "Sedimentation Amplitude (\$A_{sed}\$)",
    :conical_scale => "Conical Control"
)






# Column Superaxis Label (e.g., at the very top, centered horizontally)
fig.text(0.5, -0.02, label_dict[COL_PARAM]   , ha="center", va="bottom", fontsize=22, weight="bold")

# Row Superaxis Label (e.g., on the very left, centered vertically)
fig.text(-0.02, 0.5, label_dict[ROW_PARAM], ha="left", va="center", fontsize= 22, weight="bold", rotation=90)

# Adjust layout and display
#plt.tight_layout(rect=[0.05, 0.05, 0.9, 0.95]) # Adjust layout to make space for labels and colorbar
#plt.suptitle("Morphospace for $(ROW_PARAM) vs $(COL_PARAM)", fontsize=18, y= 1.02, weight="bold")

# --- Simplified Superaxis Arrows and Labels using fig.annotate ---

# Get the bounding box of the entire grid of subplots for better positioning
bbox_bottom_left = axes_arr[rows, 1].get_position()
bbox_top_right = axes_arr[1, cols].get_position()

grid_x_start = bbox_bottom_left.x0
grid_x_end = bbox_top_right.x1
grid_y_start = bbox_bottom_left.y0
grid_y_end = bbox_top_right.y1

# Arrow properties for annotate
arrow_props = Dict(
    "arrowstyle" => "->", # Simple arrow style
    "color" => "black",
    "lw" => 3, # Line width
    "mutation_scale" => 10 # Scale factor for the head
)

# Horizontal Arrow for Column Parameter (at the bottom-left of the grid)
# xy is the arrow tip, xytext is the text position (and arrow base)
col_arrow_tip_x = 0.1  # Tip of the arrow
col_arrow_tip_y = 1.0 # Below the grid

col_arrow_base_x = 0.1 # Base of the arrow
col_arrow_base_y =   0.1 # Below the grid




plt.tight_layout()
PyPlot.matplotlib.use("Agg")
plt.show() # Display the plot


savefilename = "test"

#plt.savefig( savefilename * ".pdf",  bbox_inches="tight", pad_inches=0.1, dpi= 150)
#plt.savefig(savefilename * ".jpeg",  bbox_inches="tight", pad_inches=0.1, dpi= 100)
#plt.savefig(savefilename * ".png",  
#            bbox_inches="tight", 
#            pad_inches=0.01, 
#            dpi=120,
#            pil_kwargs=Dict("optimize" => true, "compress_level" => 9, "metadata" => Dict())
#)
#plt.close()

PARAMETER SWEEP CODE: Code which runs over many parameter combinations in a parallel to generate quantative data using Latin Hypercube sampling

In [ ]:
GC.gc() 



#Code to loop through parameters 
@everywhere begin


#Function to colour the agents when plotting them 
function agent_color(agent)
    if agent isa Precipitate
        return :green # Color for Precipitate agents
    elseif agent isa Sediment
        return :gray # Color for Sediment agents
    else
        return :gray # Default color for any other unexpected types
    end
end





    

#Helper function
get_solid_setup(model) = model.solid

#Check number of threads, may need to restart kernel if this is not what you expect
println("Number of Threads make sure set to higher than 1 for parrellisation of parameter sweep")
println(Threads.nthreads())



    
    
#Set-up parameter range to sweep over and the sampler parameters


#N_RUNS = 1000


N_RUNS = Int(1500)
N_GENS = 1000

param_bounds = Dict(
    #:precip_amplitude         => (0.01, 0.3,  :linear, :float), #Can plot the precip frac and see where it plateua for a given removal probs to pick upper bound 
    
    :stability_dist      => (1.0, 50.0,  :linear, :int),
    
    :attraction_dist     => (1.0, 50.0,  :linear, :int)
)

    

param_keys = collect(keys(param_bounds))
k_dims = length(param_keys)



# Generate sample
# Generate optimized LHC (returns strata)
plan, _ = LHCoptim(N_RUNS, k_dims, N_GENS, threading=true)

# Create 0–1 ranges for each dimension
unit_ranges = [(0.0, 1.0) for _ in 1:k_dims]

#Scale strata → normalized matrix
normalized_design_matrix = scaleLHC(plan, unit_ranges)

param_combinations = Vector{Dict{Symbol,Union{Int,Float64}}}(undef, N_RUNS)

for i in 1:N_RUNS
    combo_dict = Dict{Symbol, Any}()

    for j in 1:k_dims
        key = param_keys[j]
        (min_val, max_val, scale, type) = param_bounds[key]
        u = normalized_design_matrix[i,j]

        if scale == :log
            # Log-space mapping
            log_min = log10(min_val)
            log_max = log10(max_val)
            value = 10^(log_min + u*(log_max - log_min))
            combo_dict[key] = value

        elseif type == :int
            # Correct integer sampling (uniform)
            value = Int(floor(min_val + u*(max_val - min_val + 1e-12)))
            combo_dict[key] = clamp(value, Int(min_val), Int(max_val))

        else
            # Linear continuous
            value = min_val + u*(max_val - min_val)
            combo_dict[key] = value
        end
    end

    param_combinations[i] = combo_dict
end
end

@everywhere begin

#Function which adds the parameters to a filename for saving     
function dict_to_filename_string(d::Dict; separator::AbstractString="_", pair_separator::AbstractString="_")
    parts = String[]
    for (key, value) in d
        # Convert key to string, ensuring no problematic characters
        key_str = replace(string(key), r"[^\w\d-]" => "") # Remove non-alphanumeric, non-hyphen chars

        # Convert value to string, handling common problematic types for filenames
        value_str = if typeof(value) <: AbstractArray || typeof(value) <: Tuple
            # For arrays/tuples, join elements with a specific character (e.g., '-', '_')
            # You might need to refine this based on the complexity of your array/tuple elements
            join(string.(value), "-")
        else
            string(value)
        end
        
        # Remove any characters from values that might cause issues in filenames
        #value_str = replace(value_str, r"[^\w\d-.]" => "") # Allow alphanumeric, hyphen, dot

        push!(parts, "$(key_str)$(pair_separator)$(value_str)")
    end
    return join(parts, separator)
end





# --- Utility function to create a stable, unique ID from parameters ---
function make_unique_id(params::AbstractDict{Symbol,<:Any})
    # Convert to sorted string for deterministic hashing
    param_str = join(sort(string.(collect(pairs(params)))), ";")

    hash_str = bytes2hex(sha1(param_str))[1:8]
    
    # Add current date and time (YYYYMMDD_HHMMSS)
    timestamp = Dates.format(now(), "yyyymmdd_HHMMSS")
    
    return "sim_" * timestamp * "_" * hash_str
end


function measure_compression_ratio(data::AbstractArray)
        # Convert the array to a raw byte array (important for compression)
        data_bytes = reinterpret(UInt8, vec(data))
        uncompressed_size = length(data_bytes)
        
        # Compress the byte array using GZip
        compressed_bytes = transcode(GzipCompressor, data_bytes)
        compressed_size = length(compressed_bytes)
        
        # Avoid division by zero if data is empty or yields a zero-length compression
        if compressed_size == 0
            return 1.0 # Or handle as error/NaN
        end
        
        # Calculate the ratio
        ratio = uncompressed_size / compressed_size
        return ratio
    end






function compute_fft_peak(x)
    N = length(x)
    window = 0.5 .- 0.5 * cos.(2π .* (0:N-1) ./ N)  # Hann window
    X = rfft((x .- mean(x)) .* window)
    mag = abs.(X) ./ N
    mag[2:end-1] .*= 2
    freqs = (0:length(mag)-1) ./ N

    # Find dominant peak
    peak_idx = argmax(mag)
    peak_freq = freqs[peak_idx]
    peak_mag = mag[peak_idx]

    # --- Compute Signal-to-Noise Ratio (SNR) ---
    # Exclude peak and its neighbors from "noise"
    exclude_window = max(1, peak_idx-1):min(length(mag), peak_idx+1)
    noise_power = mean(mag[setdiff(1:end, exclude_window)].^2)
    signal_power = peak_mag^2

    SNR = 10 * log10(signal_power / noise_power)  # in decibels

    return peak_freq, mag, SNR
end




println("Number of Combinations scanning")
println(size(param_combinations))

#Set the number of steps for the loop
num_steps = 10000    
end 



const results_lock = ReentrantLock()
const results = Vector{Dict{Symbol,Any}}()


#Setup-folder to output data to 
#output_folder_path = pwd() * "/final_quantative_sweep_2_stability_attraction"
output_folder_path = pwd() * "/test"

#Create folder if it does not exist 
mkpath(output_folder_path)




#Loop through parameters 
@showprogress Threads.@threads for p_combo in param_combinations

    #Setup paramters- onces to be scanned to called p_combo
    #Other ones can be left as default or vaired
    my_params = WorldParameters( sediment_amplitude= 70, downspeed= 3, width = 700,
    height = 1500, period_control_T=1000,   
        initial_solid_setup_type = :baseline,  sediment_periodic_mode = :max_cos ,
    precip_periodic_mode=  :max_cos     
        ;p_combo...)

    #Set-up model with or with seed 
    
    model = makeworld(my_params,  rng = Xoshiro())

   
    
    #Step the model
   step!(model, num_steps)


    object = model.solid

    compression_ratio = measure_compression_ratio(object)

    # Masks
    mask_precip = (object .== 3)
    mask_sed = (object .== 2)


    grid_size = model.width*model.height

    max_height = find_max_height(model.solid)


    

    # Convert 0/1 integer mask to Bool
    mask_solid_bool = mask_sed .| mask_precip  # or mask_solid .!= 0

    # Vectorized: find first non-zero row per column
    heights = [findfirst(reverse(col)) |> (i -> isnothing(i) ? 0 : length(col)-i+1) for col in eachcol(mask_solid_bool')] 

    mean_height = mean(heights)

    if max_height != maximum(heights)
       @warn "ERROR IN MEASUREMEMTS"

    end


    if length(heights)!=model.width
        @warn "ERROR IN Matrix conventions with sums over horizontal profile"
    end

    # roughness metrics
    roughness_std     = std(heights)
    #roughness_max_min = maximum(heights) - minimum(heights)
    local_grad        = diff(heights)
    roughness_local   = sqrt(mean(local_grad .^ 2))
    #roughness_rms     = sqrt(mean((heights .- mean_height).^2))

    

    #Filled fraction horziontal profile, measurements, taking verical slices
    #Filled fraction should be shape of model.width
    

    

    #Safe division incase a height is zero 
    @assert length(heights) == model.width "heights vector should have one entry per column (width)!"

    heights_safe = max.(heights, 1)     # replace zeros with 1 temporarily
    filled_frac = sum(mask_solid_bool, dims=2) ./ heights_safe
    filled_frac[heights .== 0] .= 0.0 
    

    filled_frac_mean = mean(filled_frac)
    
    filled_frac_std = std(filled_frac)


    filled_frac_detrended = filled_frac .- filled_frac_mean


    filled_frac_freq, mag_filled, filled_frac_SNR = compute_fft_peak(vec(filled_frac))



    

    diff_rows = diff(mask_solid_bool; dims=1)

    @assert size(diff_rows) == (model.width-1, model.height) "Horizontal diff shape mismatch"

    #println(size(diff_rows))
    n_transitions = count(!iszero, diff_rows)
    # Normalize by total rows to make it scale invariant
    horizontal_transisitions_per_area = n_transitions/ sum(mask_solid_bool)
    
    se = trues(3, 3)  # structuring element: 1 row, 3 columns (horizontal), adjust size to max gap
    mask_closed = closing(mask_solid_bool, se)


    diff_rows_closed = diff(mask_closed; dims=1)
    n_transitions_closed = count(!iszero, diff_rows_closed)
    # Normalize by total rows to make it scale invariant
    closed_horizontal_transisitions_per_area = n_transitions_closed/ sum(mask_closed)


    #Types of solid vertical profile, measurements 
    

    # Sum across rows
    precip_sum = vec(sum(mask_precip, dims=1))
    sed_sum = vec(sum(mask_sed, dims=1))



    if length(precip_sum)!=model.height
        @warn "ERROR IN Matrix conventions with sums over vertical profile"
    end

    # Compute fractions
    denom = sed_sum .+ precip_sum
    denom[denom .== 0] .= 1  # prevent division by zero
    sed_frac = sed_sum ./ denom
    precip_frac = precip_sum ./ denom








    



    
    # --- Create unique simulation ID ---
    sim_id = make_unique_id(p_combo)
    


    
    #println(model.parameters)

    # Store results with parameters
        result_entry = merge(
            deepcopy(p_combo),
        Dict(
            :sim_id => sim_id,

            #Basic Fractions
            :sed_frac_total => sum(mask_sed)/grid_size,
            :precip_frac_total => sum(mask_precip)/grid_size,
           

            #Geomtrey 
                
            :mean_height => mean_height/ model.height,
            :max_height => max_height/ model.height,


            #Roughness 
            
            :roughness_cv => roughness_std / mean_height,
            :roughness_local => roughness_local,

            #Filled Fractions
            :filled_frac_mean => filled_frac_mean,
            :filled_frac_std => filled_frac_std,
            :filled_frac_freq => filled_frac_freq,
            :filled_frac_SNR => filled_frac_SNR,
            
              #Complexity
            :gzip_compression_ratio => compression_ratio,
            
        ))

        # Lock and append safely
        lock(results_lock) do
            push!(results, result_entry)
        end
    yield()
   
   
    #fig
     plotkwargs = (
    agent_size = 3, 
    heatarray = get_solid_setup,
    heatkwargs = (colorrange = (0, 3), colormap = :gnuplot),
    agent_color = agent_color,
          
    #colorbar = false, 
    add_colorbar = false,
)

    #Save the output 
    fig, _ = abmplot(model; figure = (size = (model.width, model.height),),
          plotkwargs...)
    


   

    #If you don't want to compress of want to change data format, save matrix etc this can be changed 
     png_file = joinpath(output_folder_path, "$(sim_id).png")
    jpg_file = joinpath(output_folder_path, "$(sim_id).jpg")

    save(png_file , fig)

    hdf5_file = joinpath(output_folder_path, "$(sim_id)_solid.hdf5")
    matrix_to_save = model.solid

#Use h5open to create/open the file and write the matrix
    try h5open(hdf5_file, "w") do hf
      # 1. Define the parameters
        chunk_dims = size(matrix_to_save)
        data_type = eltype(matrix_to_save)
        data_dims = size(matrix_to_save)
        
        # 2. Create the dataset with the desired properties
        #    Note: The dataset is created empty here.
        dset = create_dataset(
            hf,                          # File handle
            "solid_data",                # Dataset name
            data_type,                   # Data type (e.g., Float64)
            data_dims,                   # Full dimensions of the dataset
            chunk = chunk_dims,          # Explicitly set chunking
            compress = 5                 # Explicitly set compression level
        )
        
        # 3. Write the actual data to the created dataset
        write(dset, matrix_to_save)

        end
    catch e
    @error "Failed to save matrix to HDF5: $e"
    end




    

    if isfile(png_file) && stat(png_file).size > 0
        try
            img = load(png_file)
           
          # 2. Resize the image (creates a specialized/view type)
            img_resized_view = imresize(img, ratio = 0.5)
            
            # 2.5. CRITICAL FIX: Convert the resized image into a basic array.
            #      This makes it compatible with the JPEG saving library.
            img_small = Array(img_resized_view)
            
            # 3. Save the now compatible image as a compressed JPEG
            #    (Note: The variable name must be consistent with what you save)
            save(jpg_file, img_small; quality=80)  
           
            rm(png_file; force=true)
        catch e
            @warn "Failed to convert PNG to JPG for $sim_id: $e"
        end
    else
        @warn "Skipping empty PNG file for $sim_id"
    
    end






    
end



    
  


GC.gc() 

The below cells contain the plotting and data analysis code for sweeping over parameters plotting some single plots ans also being able to do pairwise combinations of all plots. Some cells can be used for plots of single data objects while others can be used to sweep over the whole dataset and do pairwise plots of everything depending on the analysis required. All aesthtic and ploting choices can be changed. The sweep over all plots should run but depending on which of the above version of the sweep is run some of the single plots may fail if they are run out of order so care should be taken with this and rewritting over the df object etc. 

Important Cell: which is used to save the reuslts plotting data, this should be run.  

In [ ]:
using DataFrames, Plots, Statistics

#Plotting Metadata

df = DataFrame(results)
df = df[:, sort(names(df))]




metadata!(df, "labels", Dict(
    :precip_amplitude => "Precipitation Amplitude (\$A_{precip}\$)",
    :precip_frac_total => "Precipitate Solid Fraction",
    :attraction_dist => "Attraction Dist.",
    :filled_frac_SNR => "Filled Fraction SNR",
    :filled_frac_freq => "Filled Fraction Freq.",
    :filled_frac_mean => " Filled Fraction Mean",
    :filled_frac_std => "Filled Fraction Std. Dev.",
    :gzip_compression_ratio => "Compression Ratio",
    :max_height => "Max. Height",
    :mean_height => "Mean Height",
    :roughness_cv => "Roughness (Coefficent of Variation)",
    :roughness_local => "Roughness (Local)",
    :sed_frac_total => "Sediment Solid Fraction",
    :stability_dist => "Stability Dist."   
        
        
        
        
))



csv_path = output_folder_path * "/" * "Results.csv"



CSV.write(csv_path, df)




label(df, col) = DataFrames.metadata(df)["labels"][col]
 



println(names(df))

DataFrames.metadata(df)


In [ ]:

df = DataFrame(results)


metadata!(df, "labels", Dict(
    :precip_amplitude => "Precipitation Amplitude (\$A_{precip}\$)",
    :precip_frac_total => "Precipitate Solid Fraction",
    :attraction_dist => "Attraction Dist.",
    :filled_frac_SNR => "Filled Fraction SNR",
    :filled_frac_freq => "Filled Fraction Freq.",
    :filled_frac_mean => " Filled Fraction Mean",
    :filled_frac_std => "Filled Fraction Std. Dev.",
    :gzip_compression_ratio => "Compression Ratio",
    :max_height => "Max. Height",
    :mean_height => "Mean Height",
    :roughness_cv => "Roughness (Coefficent of Variation)",
    :roughness_local => "Roughness (Local)",
    :sed_frac_total => "Sediment Solid Fraction",
    :stability_dist => "Stability Dist."   
        
        
        
        
))






df = df[:, sort(names(df))]

#Need to set plotting backend to make sure it works
gr()

clean_label(name::Symbol) = uppercasefirst(replace(string(name), "_" => " "))
clean_label(name::AbstractString) = uppercasefirst(replace(name, "_" => " "))


# Columns chosen by symbol
xcol = :attraction_dist

ycol = :precip_frac_total

x = df[:, xcol]
y = df[:, ycol]





Plots.scatter(
    x, y,
    xlabel = clean_label(xcol),
    ylabel = clean_label(ycol),
    legend = false,
    color  = :blue,
    alpha = 0.6,
    markersize = 7,
    markerstrokecolor = :white,
    markerstrokewidth = 0.3,
    grid = false,
    framestyle = :box,
    background_color = :white,
    dpi = 200,
)




In [ ]:
using Plots, PGFPlotsX

#If you want a Latex plot can switch the backend 
#pgfplotsx()
#Plots.default(show=true)


gr()


#Add loop to save all of the plots in one go for all combinations

metadata!(df, "labels", Dict(
    :precip_amplitude => "Precipitation Amplitude (\$A_{precip}\$)",
    :precip_frac_total => "Precipitate Solid Fraction",
    :attraction_dist => "Attraction Dist.",
    :filled_frac_SNR => "Filled Fraction SNR",
    :filled_frac_freq => "Filled Fraction Freq.",
    :filled_frac_mean => " Filled Fraction Mean",
    :filled_frac_std => "Filled Fraction Std. Dev.",
    :gzip_compression_ratio => "Compression Ratio",
    :max_height => "Max. Height",
    :mean_height => "Mean Height",
    :roughness_cv => "Roughness (Coefficent of Variation)",
    :roughness_local => "Roughness (Local)",
    :sed_frac_total => "Sediment Solid Fraction",
    :stability_dist => "Stability Dist." ) ) 
        
        
        
        




clean_label(name::Symbol) = uppercasefirst(replace(string(name), "_" => " "))
clean_label(name::AbstractString) = uppercasefirst(replace(name, "_" => " "))

# Columns chosen by symbol
xcol = :attraction_dist
ycol = :max_height

x = df[:, xcol]
y = df[:, ycol]

# --- Bin the x values ---
n_bins = 20
bin_edges = range(minimum(x), stop=maximum(x), length=n_bins+1)
bin_centers = (bin_edges[1:end-1] .+ bin_edges[2:end]) ./ 2

mean_y = Float64[]
std_y = Float64[]

for i in 1:n_bins
    in_bin = (x .>= bin_edges[i]) .& (x .< bin_edges[i+1])
    y_bin = y[in_bin]
    if !isempty(y_bin)
        push!(mean_y, mean(y_bin))
        push!(std_y, std(y_bin))
    else
        push!(mean_y, NaN)
        push!(std_y, NaN)
    end
end

# --- Plot ---
Plots.plot(
    x, y,
    seriestype = :scatter,
    xlabel = label(df, xcol), 
    ylabel = label(df, ycol),
    legend = false,
    color = :blue,
    alpha = 0.3,
    markersize = 6,
    markerstrokecolor = :white,
    markerstrokewidth = 0.3,
    grid = false,
    framestyle = :box,
    background_color = :white,
    dpi = 200,
)



Plots.plot!(
    bin_centers,
    mean_y,
    yerror = std_y,           
    color = :red,
    lw = 2,
    marker = :square,
    markersize = 5,
    label = "Mean ± SD" 
)



#Plots.savefig("precip_fraction_growth.pdf")

In [ ]:


gr()


clean_label(name::Symbol) = uppercasefirst(replace(string(name), "_" => " "))
clean_label(name::AbstractString) = uppercasefirst(replace(name, "_" => " "))


# Columns chosen by symbol
xcol = :attraction_dist
ycol = :stability_dist
zcol = :mean_height

x = df[:, xcol]
y = df[:, ycol]
z = df[:, zcol]

Plots.scatter(
    x,
    y,
    zcolor = z,
    xlabel = label(df, xcol),
    ylabel = label(df, ycol),
    colorbar_title = label(df, zcol),
    markersize = 9,
    alpha = 0.85,
    markerstrokecolor = :white,
    markerstrokewidth = 0.35,
    color = :viridis,
    #title = "Relationship between $(xcol), $(ycol) and $(zcol)",
    legend = false,
    grid = false,
    framestyle = :box,
    colorbar = true,
    xticks = :auto,
    yticks = :auto,
    dpi =200
)


#Plots.savefig("stability_dist_growth_height.pdf")

In [ ]:
gr()

#df = DataFrame(results)






#df = df[:, sort(names(df))]

println(names(df))




# Columns you want at the front
priority = sort(string.(collect(keys(param_bounds))))

# All other columns, preserving their original order
rest = setdiff(names(df), priority)

# New column order: priority first, then the rest
new_order = vcat(priority, rest)

# Reorder DataFrame
df = df[:, new_order]

num_input_var = length(priority)
# Compute correlation matrix for numeric columns
numeric_df = select(df, names(df, Number))
corr_matrix = cor(Matrix(numeric_df))

n = length(names(numeric_df))

# Flattened annotation list (x, y, text)
# Build colors for ticks: first n = num_input_var in one color, rest in another
tick_colors = vcat(fill(:red, num_input_var), fill(:black, n - num_input_var))

# Flattened annotation list
annotations = []
for i in 1:n, j in 1:n
    push!(annotations, (i, j, string(round(corr_matrix[j, i]; digits=2))))
end

#clean_names = [uppercasefirst(replace(string(name), "_" => " ")) for name in names(numeric_df)]

#clean_names = [i <= num_input_var ? "* " * clean_names[i] : clean_names[i] for i in 1:length(clean_names)]


cols = Symbol.(names(numeric_df))



metadata!(df, "labels", Dict(
    :precip_amplitude => "Precipitation Amplitude (\$A_{precip}\$)",
    :precip_frac_total => "Precipitate Solid Fraction",
    :attraction_dist => "Attraction Dist.",
    :filled_frac_SNR => "Filled Fraction SNR",
    :filled_frac_freq => "Filled Fraction Freq.",
    :filled_frac_mean => " Filled Fraction Mean",
    :filled_frac_std => "Filled Fraction Std. Dev.",
    :gzip_compression_ratio => "Compression Ratio",
    :max_height => "Max. Height",
    :mean_height => "Mean Height",
    :roughness_cv => "Roughness (Coefficent of Variation)",
    :roughness_local => "Roughness (Local)",
    :sed_frac_total => "Sediment Solid Fraction",
    :stability_dist => "Stability Dist."   
        
        
        
        
))







clean_names = [label(df, col) for col in cols]

# Plot heatmap
p= Plots.heatmap(
    1:n, 1:n, corr_matrix;
    c = :coolwarm,
    aspect_ratio = :equal,
    colorbar_title = "Correlation",
    annotations = annotations,
    annotationfontsize = 10,
    annotationcolor = :black, 
    size = (3000, 3000),
    xrotation = 45,
    xticks = (1:n, clean_names),
    yticks = (1:n, clean_names),
    colorbar_titlefont = font(:bold, 16),
    #xtickfontcolor = tick_colors,   # color the x-axis labels
    #ytickfontcolor = tick_colors,   # color the y-axis labels
    xguidefont = font(:bold, 14),   # Bold x-axis label
    yguidefont = font(:bold, 14),   # Bold y-axis label
    xtickfont = font(:bold, 14),    # Bold x tick labels
    ytickfont = font(:bold, 14),    # Bold y tick labels
    titlefont = font(:bold, 20),    # Bold title
    alpha = 0.8,
    fontsize = 20
    
    
)


vline!([num_input_var + 0.5], color=:black, lw=5, linestyle=:dash, legend= false, alpha =0.8)
hline!([num_input_var + 0.5], color=:black, lw=5, linestyle=:dash,  legend= false, alpha =0.8)

Plots.xlims!(0.5, n + 0.5)
Plots.ylims!(0.5, n + 0.5)


#For powerpoint will need a .png file as pdf will blur


savepath = output_folder_path * "/" * "corr_matrix" * ".pdf"

Plots.savefig(p,savepath)

display(p)



In [ ]:
#Looped Plots for pairwsie correlations


#param_keys

using Plots, PGFPlotsX

#If you want a Latex plot can switch the backend 
#pgfplotsx()

gr()

#numeric_df
metadata!(df, "labels", Dict(
    :precip_amplitude => "Precipitation Amplitude (\$A_{precip}\$)",
    :precip_frac_total => "Precipitate Solid Fraction",
    :attraction_dist => "Attraction Dist.",
    :filled_frac_SNR => "Filled Fraction SNR",
    :filled_frac_freq => "Filled Fraction Freq.",
    :filled_frac_mean => " Filled Fraction Mean",
    :filled_frac_std => "Filled Fraction Std. Dev.",
    :gzip_compression_ratio => "Compression Ratio",
    :max_height => "Max. Height",
    :mean_height => "Mean Height",
    :roughness_cv => "Roughness (Coefficent of Variation)",
    :roughness_local => "Roughness (Local)",
    :sed_frac_total => "Sediment Solid Fraction",
    :stability_dist => "Stability Dist."   
        
        
        
        
))







for a in param_keys

    for b in Symbol.(names(numeric_df))

    if b ∉ param_keys

clean_label(name::Symbol) = uppercasefirst(replace(string(name), "_" => " "))
clean_label(name::AbstractString) = uppercasefirst(replace(name, "_" => " "))

# Columns chosen by symbol
xcol = a
ycol = b

x = df[:, xcol]
y = df[:, ycol]

# --- Bin the x values ---
n_bins = 20
bin_edges = range(minimum(x), stop=maximum(x), length=n_bins+1)
bin_centers = (bin_edges[1:end-1] .+ bin_edges[2:end]) ./ 2

mean_y = Float64[]
std_y = Float64[]

for i in 1:n_bins
    in_bin = (x .>= bin_edges[i]) .& (x .< bin_edges[i+1])
    y_bin = y[in_bin]
    if !isempty(y_bin)
        push!(mean_y, mean(y_bin))
        push!(std_y, std(y_bin))
    else
        push!(mean_y, NaN)
        push!(std_y, NaN)
    end
end

# --- Plot ---





Plots.scatter(
    x, y,
    #seriestype = :scatter,
    xlabel = label(df, xcol), 
    ylabel = label(df, ycol),
    legend = false,
    color = :blue,
    alpha = 0.3,
    markersize = 6,
    markerstrokecolor = :white,
    markerstrokewidth = 0.3,
    grid = false,
    framestyle = :box,
    background_color = :white,
    dpi = 200,
)





# Error Bars only (Middle)
Plots.plot!(
    bin_centers,
    mean_y,
    yerror = std_y,
    seriestype = :scatter, # Use scatter to keep points discrete
    markersize = 0,        # Hide the markers for now
    color = :red,
    linewidth = 2,
    label = ""
)

# Markers only (Absolute Front)
Plots.scatter!(
    bin_centers,
    mean_y,
    color = :red,
    marker = :square,
    markersize = 6,
    label = "Mean ± SD"
)




            

            


savepath = output_folder_path * "/" * string(a) * string(b) * ".pdf"

Plots.savefig(savepath)
            



        end
    end
end

        



Code which loops through to create heatmaps

In [ ]:

using Plots, PGFPlotsX

#If you want a Latex plot can switch the backend 
#pgfplotsx()
gr()


for i in 1:length(param_keys)
    for j in i+1:length(param_keys)

        a = param_keys[i]
        c = param_keys[j]


    for b in Symbol.(names(numeric_df))

    if b ∉ param_keys

    if c ≠ a 
            



# Columns chosen by symbol
xcol = a
ycol = c
zcol = b

x = df[:, xcol]
y = df[:, ycol]
z = df[:, zcol]

Plots.scatter(
    x,
    y,
    zcolor = z,
    xlabel = label(df, xcol),
    ylabel = label(df, ycol),
    colorbar_title = label(df, zcol),
    markersize = 7,
    alpha = 0.85,
    markerstrokecolor = :white,
    markerstrokewidth = 0.35,
    color = :viridis,
    colorbar_titlefont = (8, :black),
    #title = "Relationship between $(xcol), $(ycol) and $(zcol)",
    legend = false,
    grid = false,
    framestyle = :box,
    colorbar = true,
    xticks = :auto,
    yticks = :auto,
    dpi =200,
    #colorbar_titlepadding = 40
)


savepath = output_folder_path * "/" * string(a) * string(c)* string(b) * ".pdf"

Plots.savefig(savepath)
            

                end
                end
        end
    end
end

In [ ]:
using StatsPlots
using DataFrames

using PairPlots


df = df[:, sort(names(df))]

println(names(df))




# Columns you want at the front
priority = sort(string.(collect(keys(param_bounds))))

# All other columns, preserving their original order
rest = setdiff(names(df), priority)

# New column order: priority first, then the rest
new_order = vcat(priority, rest)

# Reorder DataFrame

df = df[:, new_order]

numeric_cols = names(df, Number) 

df_pair = df[:, numeric_cols]

  # automatically pick numeric columns
fig = pairplot(df_pair  =>  (PairPlots.Scatter(markersize = 10),
        PairPlots.Contour(color=:black),
        PairPlots.MarginHist(),
        PairPlots.TrendLine(color=:red),
        PairPlots.PearsonCorrelation(),
        PairPlots.MarginDensity()))

#save("test_pair_plot.png", fig)

Code to load data from HDF5 and check it works.

In [ ]:

using HDF5

# Recreate the file path used during the saving step
# Ensure this path matches where your file was saved
hdf5_file = "/Users/nrodger3/Library/CloudStorage/OneDrive-UniversityofEdinburgh/Postdoc/Stromatalites/test_frequency_data_6/sim_20251114_152705_94f4db68_solid.hdf5"

println("Attempting to load data from: $hdf5_file")


loaded_data = nothing

if isfile(hdf5_file)
    try
        h5open(hdf5_file, "r") do hf
            
            # The name 'solid_data' must match the dataset name used during creation (create_dataset)
            global loaded_data = read(hf, "solid_data")
            
            # --- Inspection ---
            println("\n--- HDF5 Load Successful ---")
            println("File: $hdf5_file")
            println("Data Type: **$(typeof(loaded_data))**")
            println("Dimensions: **$(size(loaded_data))**")
            
            # Optional: Check the first element to confirm the values are correct
            println("First element value: $(loaded_data[begin])") 
        end
        
    catch e
        println("\n--- HDF5 Load Failed ---")
        println("An error occurred during loading: $e")
        println("Tip: If the file exists, check the **dataset name** ('solid_data') for typos.")
    end
else
    println("\nERROR: HDF5 file not found at the specified path: $hdf5_file")
end


#print(loaded_data)
solid_data = loaded_data

# Assuming 'solid_data' is your matrix
min_val = minimum(solid_data) 
max_val = maximum(solid_data) 

# Use the heatmap function
# Note: Plots.jl heatmap typically plots (columns, rows), 
# so using transpose(solid_data) is generally the correct way 
# to orient it to match the visual intuition of (x, y) coordinates.
# The 'clims' argument handles vmin/vmax.




Plots.heatmap(
    transpose(solid_data), 
    #title = plot_title,
    #xlabel = x_label,
    #ylabel = y_label,
    c = :gnuplot,           #  Sets the 'gnuplot' colormap (c is alias for colormap)
    clims = (min_val, max_val), # ⬅️Sets the value limits (clims is alias for clim)
    #aspect_ratio = :equal,      # Ensures cells are square
    #colorbar_title = "Value"
)



